In [42]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType


In [43]:
spark = SparkSession.builder \
 .appName("FraudDetection") \
 .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
 .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [44]:
users_df=spark.read.csv("data/user_table.csv",header=True,inferSchema=True)

In [45]:
tx_schema = StructType([
 StructField("tx_id", IntegerType(), True),
 StructField("userId", IntegerType(), True),
 StructField("amount", DoubleType(), True),
 StructField("timestamp", DoubleType(), True)
])

kafka_stream = spark.readStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("subscribe", "fraud-detection") \
 .load()


In [46]:
parsed_stream=kafka_stream.select(from_json(col("value").cast("string"),
                                            tx_schema).alias("tx")).select("tx.*")

In [47]:
fraud_stream=parsed_stream.filter(col("amount")>10000.0)
fraud_stream1=parsed_stream.filter(col("amount")>5000.0)

In [48]:
enriched_fraud=fraud_stream.join(users_df,"userId")
enriched_fraud1=fraud_stream1.join(users_df,"userId")

In [49]:
output_stream=enriched_fraud \
.withColumn("value",to_json(struct("*")).cast("string")) \
.select("value")
output_stream1=enriched_fraud1 \
.withColumn("value",to_json(struct("*")).cast("string")) \
.select("value")

In [ ]:
query = output_stream.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification") \
 .option("checkpointLocation", "/workspace/checkpoints") \
 .start()
query1 = output_stream1.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "read-notification") \
 .option("checkpointLocation", "/workspace/checkpoints") \
 .start()

query.awaitTermination()
query1.awaitTermination()

26/06/19 06:10:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:10:02 WARN StreamingQueryManager: Stopping existing streaming query [id=9279ac0b-0db5-4947-8444-11b5e9f405a3, runId=52d0a211-3ede-46ba-9a80-a248671bf716], as a new run is being started.
26/06/19 06:10:02 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:10:02 WARN StreamingQueryManager: Stopping existing streaming query [id=9279ac0b-0db5-4947-8444-11b5e9f405a3, runId=ce316b82-bee4-40e3-aad3-706e043660db], as a new run is being started.
26/06/19 06:10:02 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                